### 2. Initialize the `bert-base-uncased` tokenizer

Initialize the `bert-base-uncased` tokenizer.

**Question:** What is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?

In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print("Vocab size: ", tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Vocab size:  30522


### 3. Retrieve the `[SEP]` token ID

Using the `bert-base-uncased` tokenizer from the previous step, extract the exact integer ID assigned to the `[SEP]` (Separator) token.

In [6]:
sep_id = tokenizer.sep_token_id
print("[SEP] token id:", sep_id)

[SEP] token id: 102


### 4. Tokenize the `prompt` column

Using the `bert-base-uncased` tokenizer, tokenize the entire `prompt` column of the train dataset simultaneously with the following settings:

- `padding="max_length"`
- `truncation=True`
- `max_length=128`
- `return_tensors="pt"`

**Question:** What is the exact shape (dimensions) of the resulting `input_ids` tensor?

In [7]:
prompts = list(train_ds["prompt"])

encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("input_ids shape:", encoded["input_ids"].shape)

input_ids shape: torch.Size([2000, 128])


## BERT/RoBERTa Architecture & Attention Mechanisms


### 5. Compute the attention head size

A standard `bert-base-uncased` model has a hidden embedding size of `768` and uses exactly `12` attention heads in each layer.

In the Transformer architecture, the hidden size is divided equally among the attention heads.

**Question:** What is the exact dimensionality (size) of each individual attention head?

In [8]:
hidden_size = 768
num_heads = 12
head_dim = hidden_size // num_heads
print("Each attention head size:", head_dim)


Each attention head size: 64


### 6. Obtain the `last_hidden_state`

Load the `bert-base-uncased` model using `AutoModel.from_pretrained()`.

Tokenize the `prompt` from row ID `0` using the tokenizer's default settings (do **not** apply any manual padding or truncation). Pass the tokenized input through the model.

**Question:** What is the exact shape of the returned `last_hidden_state` tensor?

> **Note:** Zero-indexing is used.

In [9]:
model = AutoModel.from_pretrained("bert-base-uncased")

model.eval()
row0_prompt = train_ds[0]["prompt"]
inputs = tokenizer(row0_prompt, return_tensors='pt')

with torch.no_grad():
  outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state
print("last_hidden_state shape:", last_hidden_state.shape)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


last_hidden_state shape: torch.Size([1, 31, 768])


### 7. Extract the `[CLS]` embedding

Using the `last_hidden_state` tensor from the previous step, extract the embedding vector corresponding to the `[CLS]` token (the token at index `0`).

**Question:** What is the sum of the first `5` float values in the `[CLS]` embedding vector? Round the answer to **4 decimal places**.

In [10]:
cls_vector = last_hidden_state[0, 0, :]
cls_sum_first5 = cls_vector[:5].sum().item()
print("[CLS] first 5 values:", cls_vector[:5].tolist())
print("Sum of first 5 values (4 dp):", round(cls_sum_first5, 4))

[CLS] first 5 values: [-0.4676640033721924, -0.07544449716806412, -0.20190097391605377, -0.007064145058393478, -0.4480222761631012]
Sum of first 5 values (4 dp): -1.2001


### 8. Extract an attention weight

Load `bert-base-uncased` with `output_attentions=True`.

Tokenize the exact string `"Light-ion fusion is a technique."` using `return_tensors="pt"` and pass it through the model.

Extract the attention matrix for the **last layer** (index `-1`) and the **first attention head** (head index `0`).

**Question:** What is the exact attention weight (a float value) that the `[CLS]` token (token index `0`) assigns to the word `fusion`? Determine the token index of `fusion` from the `input_ids`, then report the corresponding attention weight rounded to **4 decimal places**.

In [11]:
attn_model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
attn_model.eval()

sentence = "Light-ion fusion is a technique."
attn_inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    attn_outputs = attn_model(**attn_inputs)

last_layer_attn = attn_outputs.attentions[-1]
attn_matrix = last_layer_attn[0, 0]

# Find the token index for "fusion"
tokens = tokenizer.convert_ids_to_tokens(attn_inputs["input_ids"][0])
print("Tokens:", tokens)

fusion_idx = tokens.index("fusion")
print("Index of 'fusion':", fusion_idx)

cls_to_fusion = attn_matrix[0, fusion_idx].item()
print("Attention weight CLS -> fusion (4 dp):", round(cls_to_fusion, 4))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Index of 'fusion': 4
Attention weight CLS -> fusion (4 dp): 0.1025


## Context-Aware Embeddings

### 9. Generate sentence embeddings and compute cosine similarity

Initialize the `sentence-transformers/all-MiniLM-L6-v2` model.

Use the model's `.encode()` method to generate embeddings for both the `prompt` and **Option B** from row ID `0`. Compute the cosine similarity between these two embeddings using `sentence_transformers.util.cos_sim()`.

**Question:** What is the resulting cosine similarity score, rounded to **4 decimal places**?

> **Note:** Zero-indexing is used.

In [14]:
minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

row0 = train_ds[0]
emb_prompt = minilm.encode(row0["prompt"], convert_to_tensor=True)
emb_optionB = minilm.encode(row0["B"], convert_to_tensor=True)

similarity = util.cos_sim(emb_prompt, emb_optionB)
print("Cosine similarity (prompt vs Option B, row 0):", round(similarity.item(), 4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Cosine similarity (prompt vs Option B, row 0): 0.7658


### 10. Compare the TF-IDF and MiniLM ranking pipelines

Build and evaluate two ranking pipelines using every row in `train.csv`.

### Pipeline 1
Use the **TF-IDF cosine similarity** approach from **Milestone 1**.

### Pipeline 2
Use the `sentence-transformers/all-MiniLM-L6-v2` model to generate embeddings for the `prompt` and all five answer options. Rank the options by cosine similarity and produce **Top-3** predictions.

**Questions:**

1. What is the final **MAP@3** score of the `all-MiniLM-L6-v2` pipeline on the entire training set?
2. How many questions have the correct answer **absent** from the TF-IDF Top-3 predictions **but present** in the MiniLM Top-3 predictions?

In [18]:
options = ["A", "B", "C", "D", "E"]

def tfidf_top3(row):
    texts = [row["prompt"]] + [row[o] for o in options]
    vec = TfidfVectorizer().fit_transform(texts)
    prompt_vec, option_vecs = vec[0], vec[1:]
    sims = cosine_similarity(prompt_vec, option_vecs)[0]
    ranked = [options[i] for i in np.argsort(sims)[::-1]]
    return ranked[:3]

def minilm_top3(row, prompt_emb, option_embs):
    sims = util.cos_sim(prompt_emb, option_embs)[0].numpy()
    ranked = [options[i] for i in np.argsort(sims)[::-1]]
    return ranked[:3]

def average_precision_at_3(top3, correct):
    if correct in top3:
        rank = top3.index(correct) + 1
        return 1.0 / rank
    return 0.0


In [19]:
all_prompts = train_ds["prompt"]
all_option_texts = {o: train_ds[o] for o in options}

prompt_embs = minilm.encode(all_prompts, convert_to_tensor=True, show_progress_bar=True)
option_embs = {o: minilm.encode(all_option_texts[o], convert_to_tensor=True, show_progress_bar=True) for o in options}


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

In [22]:
ap_scores = []
overlap_count = 0

for i, row in enumerate(train_ds):
  correct = row["answer"]

  # TF-IDF pipeline
  tfidf_ranked = tfidf_top3(row)

  # MiniLM pipeline (using pre-computed embeddings)
  row_option_embs = torch.stack([option_embs[o][i] for o in options])
  sims = util.cos_sim(prompt_embs[i], row_option_embs)[0].numpy()
  minilm_ranked = [options[j] for j in np.argsort(sims)[::-1]][:3]

  ap_scores.append(average_precision_at_3(minilm_ranked, correct))

  if (correct not in tfidf_ranked) and (correct in minilm_ranked):
    overlap_count += 1

map_at_3 = np.mean(ap_scores)
print("MAP@3 (MiniLM pipeline):", round(map_at_3, 4))
print("Questions fixed by MiniLM but missed by TF-IDF:", overlap_count)



MAP@3 (MiniLM pipeline): 0.4231
Questions fixed by MiniLM but missed by TF-IDF: 564


## Zero-shot classification concepts


### 11. Perform zero-shot classification

Initialize the Hugging Face pipeline for `"zero-shot-classification"` (which defaults to `facebook/bart-large-mnli`).

For the prompt in the **2nd row** (index `1`), use **Options A, B, and C** as the `candidate_labels`.

**Question:** What is the probability score assigned to the top-ranked option? Round the answer to **4 decimal places**.

> **Note:** Zero-indexing is used.

In [23]:
zero_shot = pipeline("zero-shot-classification")

row1 = train_ds[1]
candidate_labels = [row1["A"], row1["B"], row1["C"]]

result = zero_shot(row1["prompt"], candidate_labels)
top_score = result["scores"][0]
print("Top label:", result["labels"][0])
print("Top-ranked probability (4 dp):", round(top_score, 4))


[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Top label: Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.
Top-ranked probability (4 dp): 0.4575


### 12. Compare Softmax and Sigmoid probabilities

Run the same zero-shot classification as in the previous step, but set `multi_label=True`.

**Question:** What is the absolute difference between:

- the sum of the three probabilities from the previous step (using **Softmax**), and
- the sum of the three probabilities from this step (using **independent Sigmoids**)?

Report the exact value.

In [24]:
result_multi = zero_shot(row1["prompt"], candidate_labels, multi_label=True)

softmax_sum = sum(result["scores"][:3])
sigmoid_sum = sum(result_multi["scores"][:3])

diff = abs(softmax_sum - sigmoid_sum)
print("Sum of softmax probabilities:", round(softmax_sum, 4))
print("Sum of sigmoid probabilities:", round(sigmoid_sum, 4))
print("Absolute difference:", round(diff, 4))


Sum of softmax probabilities: 1.0
Sum of sigmoid probabilities: 0.0005
Absolute difference: 0.9995


### 13. Generate an answer using a Small Language Model

Load `google/flan-t5-small` using the Hugging Face `pipeline("text2text-generation")`.

For row index `0`, construct the following input string:

```text
Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B.
```

Pass this string to the pipeline with `max_new_tokens=5`.

**Question:** What is the exact string output returned by the model?

> **Note:** Zero-indexing is used.

In [26]:
qa_pipe = pipeline("text-generation", model="google/flan-t5-small")

row0 = train_ds[0]
prompt_text = (
    f"Question: {row0['prompt']}. "
    f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
    f"Answer with just the letter A or B."
)

output = qa_pipe(prompt_text, max_new_tokens=5)
print("Model output:", output[0]["generated_text"])

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

Model output: Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.
